# v3 그리드 — 모델별 {base · v2 · v3 · v2+v3} 선택 + 결정 게이트
기존 조합 그리드(nn-fe 후반)를 확장: 트리는 `{base, v2, v3, v2+v3}`, 선형/NN은 `{base, ratio, v3, ratio+v3}` 중 **모델별 독립 선택**.

**핵심 결정 게이트:** `best(v3 허용)` vs `best(v3 불허 = base/v2/ratio만)` 의 블렌드 OOF 증분 paired-bootstrap CI가 0 제외. ← 이게 "**v2가 깔린 운영 블렌드 위에서 v3가 이득인가**"의 직접 답. base→v3가 아니라 v2+v3 vs v2.

**입력(Add Input) — 있는 것만 자동 사용:**
- 커밋: `oof_predictions`·`test_predictions`(base 트리+lr) · `oof_v2_trees`·`test_v2_trees` · `oof_lin_ratio`·`test_lin_ratio` · `oof_nn`·`test_nn`(nn_base) · `oof_nn_ratio`(nn_ratio)
- B 산출: `oof_v3_trees`·`test_v3_trees` · `oof_v2v3_trees`·`test_v2v3_trees` · `oof_lin_v3`·`test_lin_v3` · `oof_lin_ratio_v3`·`test_lin_ratio_v3`
- A 산출(선택, nn이 v3 통과 시): `oof_nn_v3_member`·`test_nn_v3_member`(nn_v3) · `oof_nn_ratio_v3`·`test_nn_ratio_v3`(nn_ratio_v3)

## 0. 멤버 로드 (확장 MAP — 있는 파일만)

In [7]:
import os,glob,json,warnings
import numpy as np, pandas as pd
from itertools import product
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
warnings.filterwarnings("ignore")
# 입력 파일 인덱스 1회 구축 (재귀하되 catboost_info 등 잡폴더 가지치기 — 깊은 노트북 경로도 잡고 빠름)
_FILE_INDEX={}
def _build_index():
    import os
    for root in ["/kaggle/input","/kaggle/working","."]:
        if not os.path.isdir(root): continue
        for dp,dn,fn in os.walk(root):
            dn[:]=[d for d in dn if d not in ("catboost_info",".virtual_documents",".git","__pycache__")]
            for f in fn:
                if f.endswith((".csv",".json")) and f not in _FILE_INDEX:
                    _FILE_INDEX[f]=os.path.join(dp,f)
_build_index()
def find(name): return _FILE_INDEX.get(name)
train=pd.read_csv(find("train.csv")); test=pd.read_csv(find("test.csv")) if find("test.csv") else None
TARGET="임신 성공 여부"; ID_COL="ID"; y=train[TARGET].astype(int).values; N=len(train)
def runr(x): return rankdata(x)/len(x)
# DRYRUN 산출(_DRYRUN 접미사)도 자동 인식 — 단, 폴백 시 반드시 경고(제출 금지 신호)
DRYRUN_USED=[]
def fchoose(name):
    p=find(name)
    if p: return p
    pd_=find(name.replace(".csv","_DRYRUN.csv"))
    if pd_: print(f"🔴 DRYRUN 아티팩트 사용: {os.path.basename(pd_)} — 덜 학습된 멤버! 제출 금지, 풀런 산출로 교체 확인"); DRYRUN_USED.append(os.path.basename(pd_))
    return pd_
OOF_MAP=[("oof_predictions.csv",{"oof_lgb":"lgb_base","oof_cat":"cat_base","oof_xgb":"xgb_base","oof_lr":"lin_base"}),
         ("oof_v2_trees.csv",{"v2_lgb":"lgb_v2","v2_cat":"cat_v2","v2_xgb":"xgb_v2"}),
         ("oof_lin_ratio.csv",{"oof_lin_ratio":"lin_ratio"}),
         ("oof_nn.csv",{"oof_nn":"nn_base"}),("oof_nn_ratio.csv",{"oof_nn":"nn_ratio"}),
         ("oof_nn_views.csv",{"nn_base":"nn_base","nn_ratio":"nn_ratio","nn_v3":"nn_v3","nn_ratio_v3":"nn_ratio_v3"}),  # A: config-일관 4뷰(있으면 커밋 nn override)
         ("oof_v3_trees.csv",{"oof_v3_lgb":"lgb_v3","oof_v3_cat":"cat_v3","oof_v3_xgb":"xgb_v3"}),
         ("oof_v2v3_trees.csv",{"oof_v2v3_lgb":"lgb_v2v3","oof_v2v3_cat":"cat_v2v3","oof_v2v3_xgb":"xgb_v2v3"}),
         ("oof_lin_v3.csv",{"oof_lin_v3":"lin_v3"}),("oof_lin_ratio_v3.csv",{"oof_lin_ratio_v3":"lin_ratio_v3"}),
         ("oof_nn_ratio_v3.csv",{"oof_nn_ratio_v3":"nn_ratio_v3"})]   # nn_v3/nn_ratio_v3는 oof_nn_views.csv에서 로드(아래)
TST_MAP=[("test_predictions.csv",{"test_lgb":"lgb_base","test_cat":"cat_base","test_xgb":"xgb_base","test_lr":"lin_base"}),
         ("test_v2_trees.csv",{"v2_lgb":"lgb_v2","v2_cat":"cat_v2","v2_xgb":"xgb_v2"}),
         ("test_lin_ratio.csv",{"lin_ratio":"lin_ratio"}),("test_nn.csv",{"test_nn":"nn_base"}),
         ("test_v3_trees.csv",{"v3_lgb":"lgb_v3","v3_cat":"cat_v3","v3_xgb":"xgb_v3"}),
         ("test_v2v3_trees.csv",{"v2v3_lgb":"lgb_v2v3","v2v3_cat":"cat_v2v3","v2v3_xgb":"xgb_v2v3"}),
         ("test_lin_v3.csv",{"lin_v3":"lin_v3"}),("test_lin_ratio_v3.csv",{"lin_ratio_v3":"lin_ratio_v3"}),
         ("test_nn_views.csv",{"nn_base":"nn_base","nn_ratio":"nn_ratio","nn_v3":"nn_v3","nn_ratio_v3":"nn_ratio_v3"})]
OOF={}; TST={}; align_warn=[]
for fn,cmap in OOF_MAP:
    p=fchoose(fn)
    if not p: continue
    d=pd.read_csv(p)
    if len(d)!=N: print(f"  ⚠️ {fn} 길이 {len(d)}≠{N} skip"); continue
    if "y" in d.columns and not np.array_equal(d["y"].astype(int).values,y): align_warn.append(fn)
    for col,key in cmap.items():
        if col in d.columns: OOF[key]=d[col].values
for fn,cmap in TST_MAP:
    p=fchoose(fn)
    if not p: continue
    d=pd.read_csv(p)
    for col,key in cmap.items():
        if col in d.columns: TST[key]=d[col].values
print("로드 OOF 멤버:",sorted(OOF)); print("로드 test 멤버:",sorted(TST))
if align_warn: print("🔴 정렬 경고(저장 y≠train y):",align_warn,"— fold 정렬 확인!")
if DRYRUN_USED: print(f"🔴🔴 DRYRUN 멤버 {len(DRYRUN_USED)}개 사용 중 — 이 그리드 제출은 무효(풀런 산출로 다시). 목록:",DRYRUN_USED)

로드 OOF 멤버: ['cat_base', 'cat_v2', 'cat_v2v3', 'cat_v3', 'lgb_base', 'lgb_v2', 'lgb_v2v3', 'lgb_v3', 'lin_base', 'lin_ratio', 'lin_ratio_v3', 'lin_v3', 'nn_base', 'nn_ratio', 'xgb_base', 'xgb_v2', 'xgb_v2v3', 'xgb_v3']
로드 test 멤버: ['cat_v2', 'cat_v2v3', 'cat_v3', 'lgb_v2', 'lgb_v2v3', 'lgb_v3', 'lin_ratio', 'lin_ratio_v3', 'lin_v3', 'nn_base', 'xgb_v2', 'xgb_v2v3', 'xgb_v3']


## 1. 정렬 sanity (base 3트리 ≈ 0.740)

In [8]:
def hill(d,yy,n=120):
    nm=list(d); s0={k:roc_auc_score(yy,d[k]) for k in nm}; b=max(s0,key=s0.get); ens=[b]; s=d[b].copy(); best=(list(ens),s0[b])
    for _ in range(n):
        cb,ca=None,-1
        for k in nm:
            a=roc_auc_score(yy,(s+d[k])/(len(ens)+1))
            if a>ca: ca,cb=a,k
        ens.append(cb); s=s+d[cb]
        if ca>best[1]: best=(list(ens),ca)
    from collections import Counter; c=Counter(best[0]); return {k:c.get(k,0)/len(best[0]) for k in nm}, best[1]
need=["lgb_base","cat_base","xgb_base"]; assert all(k in OOF for k in need), "base 트리 OOF 필수(oof_predictions.csv)"
_,a3=hill({k.split("_")[0]:runr(OOF[k]) for k in need},y)
print(f"base 3트리 블렌드 OOF={a3:.5f} →", "✅ 정렬 정상" if a3>=0.735 else "🔴 0.735 미만 — fold 정렬 의심!")

base 3트리 블렌드 OOF=0.74012 → ✅ 정렬 정상


## 2. 모델별 뷰 선택 그리드 (4뷰) + 운영(v3 불허) 기준선

In [9]:
TREE_ALTS=["v2","v3","v2v3"]; LIN_ALTS=["ratio","v3","ratio_v3"]; NN_ALTS=["ratio","v3","ratio_v3"]
def vers(m,alts): return ["base"]+[v for v in alts if f"{m}_{v}" in OOF]
VIEWS={"lgb":vers("lgb",TREE_ALTS),"xgb":vers("xgb",TREE_ALTS),"cat":vers("cat",TREE_ALTS),"lin":vers("lin",LIN_ALTS)}
if "nn_base" in OOF: VIEWS["nn"]=vers("nn",NN_ALTS)
MODELS=list(VIEWS); NO_V3={"base","v2","ratio"}
print("가용 뷰:",{m:VIEWS[m] for m in MODELS})
RUNR={k:runr(OOF[k]) for k in OOF}                          # ★ runr 1회 캐싱 (조합마다 재계산 금지)
def blend_sel(sel):                                         # sel: {model:view}
    d={m:RUNR[f"{m}_{sel[m]}"] for m in sel}; w,a=hill(d,y); p=sum(w[k]*d[k] for k in w if w[k]>0); return w,a,p
def coord_ascent(allowed):                                  # 모델별 뷰 좌표상승 (전수 product 대신, 256k 확장성)
    sel={m:"base" for m in MODELS}                          # base에서 시작
    _,cur,_=blend_sel(sel); path=[("init",dict(sel),cur)]
    improved=True
    while improved:
        improved=False
        for m in MODELS:
            best_v,best_a=sel[m],cur
            for v in VIEWS[m]:
                if v not in allowed: continue
                t=dict(sel); t[m]=v; _,a,_=blend_sel(t)
                if a>best_a+1e-6: best_a,best_v=a,v
            if best_v!=sel[m]: sel[m]=best_v; cur=best_a; improved=True; path.append((m,dict(sel),cur))
    return sel,cur,path
ALL_VIEWS=set(sum(VIEWS.values(),[]))
sel_all,oof_all,path=coord_ascent(ALL_VIEWS)               # v3 허용
sel_no ,oof_no ,_  =coord_ascent(NO_V3)                    # v3 불허(운영급)
print(f"\nbest(v3 허용)  OOF={oof_all:.5f}  {sel_all}")
print(f"best(v3 불허) OOF={oof_no:.5f}  {sel_no}")
print(f"OOF 상 v3 상승폭 = {oof_all-oof_no:+.5f}")
print("\n좌표상승 경로(v3 허용):")
for m,s,a in path: print(f"  {m:5} → OOF={a:.5f}  {s}")

가용 뷰: {'lgb': ['base', 'v2', 'v3', 'v2v3'], 'xgb': ['base', 'v2', 'v3', 'v2v3'], 'cat': ['base', 'v2', 'v3', 'v2v3'], 'lin': ['base', 'ratio', 'v3', 'ratio_v3'], 'nn': ['base', 'ratio']}

best(v3 허용)  OOF=0.74067  {'lgb': 'v2v3', 'xgb': 'v3', 'cat': 'v2v3', 'lin': 'ratio', 'nn': 'ratio'}
best(v3 불허) OOF=0.74059  {'lgb': 'v2', 'xgb': 'base', 'cat': 'v2', 'lin': 'ratio', 'nn': 'ratio'}
OOF 상 v3 상승폭 = +0.00009

좌표상승 경로(v3 허용):
  init  → OOF=0.74045  {'lgb': 'base', 'xgb': 'base', 'cat': 'base', 'lin': 'base', 'nn': 'base'}
  lgb   → OOF=0.74060  {'lgb': 'v2v3', 'xgb': 'base', 'cat': 'base', 'lin': 'base', 'nn': 'base'}
  xgb   → OOF=0.74062  {'lgb': 'v2v3', 'xgb': 'v3', 'cat': 'base', 'lin': 'base', 'nn': 'base'}
  cat   → OOF=0.74066  {'lgb': 'v2v3', 'xgb': 'v3', 'cat': 'v2v3', 'lin': 'base', 'nn': 'base'}
  lin   → OOF=0.74067  {'lgb': 'v2v3', 'xgb': 'v3', 'cat': 'v2v3', 'lin': 'ratio', 'nn': 'base'}
  nn    → OOF=0.74067  {'lgb': 'v2v3', 'xgb': 'v3', 'cat': 'v2v3', 'lin': 'ratio', 'nn'

## 3. 결정 게이트 — best(v3) vs best(v3 불허) 증분 paired-bootstrap CI

In [10]:
def pred_sel(sel): return blend_sel(sel)[2]
pA=pred_sel(sel_all); pB=pred_sel(sel_no)
if abs(roc_auc_score(y,pA)-roc_auc_score(y,pB))<1e-9 and sel_all==sel_no:
    print("v3 허용/불허 best 동일 → v3가 운영 위에서 추가 이득 없음 (게이트 ❌, v2/ratio에 흡수).")
    GATE=False; inc,lo,hi=0.0,0.0,0.0
else:
    rng=np.random.default_rng(0); ds=np.empty(2000)
    for i in range(2000):
        ix=rng.integers(0,N,N); ds[i]=roc_auc_score(y[ix],pA[ix])-roc_auc_score(y[ix],pB[ix])
    inc,lo,hi=float(ds.mean()),float(np.percentile(ds,2.5)),float(np.percentile(ds,97.5)); GATE=lo>0
    print(f"v3 증분(운영 위) = {inc:+.5f} | 95% CI [{lo:+.5f},{hi:+.5f}] → {'✅ 채택(v3가 v2/ratio 위에서도 기여)' if GATE else '❌ 미달(v3는 v2/ratio에 흡수)'}")
print("  ⚠️ winner's curse(비대칭): v3 허용 쪽은 고를 뷰가 더 많아(base/v2/v3/v2v3) OOF 과적합 기회↑.")
print("     부트스트랩은 '선택된 뷰 고정 후 행만' 리샘플 → 이 선택 낙관은 못 잡음. CI 통과=필요조건일 뿐.")
print("     반드시 (a)멀티시드 멤버(seed 7/2024) 재확인 + (b)LB 최종심판으로 닫을 것.")

v3 증분(운영 위) = +0.00008 | 95% CI [-0.00001,+0.00019] → ❌ 미달(v3는 v2/ratio에 흡수)
  ⚠️ winner's curse(비대칭): v3 허용 쪽은 고를 뷰가 더 많아(base/v2/v3/v2v3) OOF 과적합 기회↑.
     부트스트랩은 '선택된 뷰 고정 후 행만' 리샘플 → 이 선택 낙관은 못 잡음. CI 통과=필요조건일 뿐.
     반드시 (a)멀티시드 멤버(seed 7/2024) 재확인 + (b)LB 최종심판으로 닫을 것.


## 4. 최종 제출 (best 조합, test 가용 폴백)

In [11]:
def tkey(sel):
    return {m:f"{m}_{sel[m]}" for m in sel}
# 게이트 ❌면 v3 끼우지 않음(OOF 노이즈 추격·제출 낭비 방지) → 운영급 best 유지
sel=dict(sel_all if GATE else sel_no)
print(f"게이트 {'✅ 통과 → sel_all(v3 포함) 제출' if GATE else '❌ 미달 → sel_no(운영급 v2/ratio) 유지'}: {sel}")
if not GATE: print("  ※ sel_no는 이미 운영 블렌드 계열 — v3 미채택이면 신규 제출 자체가 불필요할 수 있음(제출 1회 보호).")
tk=tkey(sel); miss=[k for k in tk.values() if k not in TST]
if miss:
    print(f"⚠️ best(v3허용) test 부재 {miss} → test 가용으로 뷰 다운그레이드")
    for m in MODELS:                                       # 부족 멤버는 test 있는 뷰로 교체(base 우선)
        if tk[m] not in TST:
            for v in ["base"]+VIEWS[m]:
                if f"{m}_{v}" in TST: sel[m]=v; break
    tk=tkey(sel)
w,a,_=blend_sel(sel)
if DRYRUN_USED:
    print(f"\n제출 조합(참고) {sel} | 블렌드 OOF={a:.5f} — 🔴 DRYRUN 멤버 포함이라 submission 파일 미생성. 풀런 후 재실행.")
elif all(k in TST for k in tk.values()):
    ptest=sum(w[m]*runr(TST[tk[m]]) for m in w if w[m]>0 and tk[m] in TST)
    sp=find("sample_submission.csv")
    if sp: s=pd.read_csv(sp); pc=[c for c in s.columns if c!=ID_COL][0]; s[pc]=ptest
    else: s=pd.DataFrame({ID_COL:test[ID_COL] if test is not None else np.arange(len(ptest)),"probability":ptest})
    s.to_csv("/kaggle/working/submission_v3_grid.csv",index=False)
    print(f"\n제출 조합 {sel} | 블렌드 OOF={a:.5f} | 가중치={ {k:round(v,2) for k,v in w.items() if v>0} }")
    print("💾 submission_v3_grid.csv → LB")
else:
    print("⚠️ test 멤버 부족 — 제출 skip:",[k for k in tk.values() if k not in TST])
json.dump({"sel_v3":sel_all,"sel_no_v3":sel_no,"oof_v3":oof_all,"oof_no_v3":oof_no,"gate":bool(GATE),"inc":inc,"ci":[lo,hi]},open("/kaggle/working/v3_grid_decision.json","w"),ensure_ascii=False,default=float)
print("💾 v3_grid_decision.json")

게이트 ❌ 미달 → sel_no(운영급 v2/ratio) 유지: {'lgb': 'v2', 'xgb': 'base', 'cat': 'v2', 'lin': 'ratio', 'nn': 'ratio'}
  ※ sel_no는 이미 운영 블렌드 계열 — v3 미채택이면 신규 제출 자체가 불필요할 수 있음(제출 1회 보호).
⚠️ best(v3허용) test 부재 ['xgb_base', 'nn_ratio'] → test 가용으로 뷰 다운그레이드

제출 조합 {'lgb': 'v2', 'xgb': 'v2', 'cat': 'v2', 'lin': 'ratio', 'nn': 'base'} | 블렌드 OOF=0.74056 | 가중치={'lgb': 0.27, 'xgb': 0.17, 'cat': 0.3, 'lin': 0.03, 'nn': 0.23}
💾 submission_v3_grid.csv → LB
💾 v3_grid_decision.json


## 5. [추가] v3 강제 제출 파일 — LB 시험용 (게이트 ❌ 무관)
게이트는 채택을 막았지만, +0.00009가 진짜 노이즈인지 **LB 1회로 직접 확인**. v3 조합을 test 가용 뷰로 저장 → 이 파일만 제출해 0.74209 초과 여부 확인.

In [12]:
# ── [추가] v3 강제 제출 파일 (게이트 ❌ 무관) — LB로 v3가 진짜 오르는지 1회 시험용
# best(v3 허용) 조합을 test 가용 뷰로 다운그레이드해 저장. sel_no(운영급)와 별개 파일.
sel_v3=dict(sel_all)
tk_v3=tkey(sel_v3); miss_v3=[k for k in tk_v3.values() if k not in TST]
if miss_v3:
    print(f"v3 강제: test 부재 {miss_v3} → 가용 뷰로 다운그레이드(base 우선, 단 base 부재 시 v2)")
    for m in MODELS:
        if tk_v3[m] not in TST:
            for v in ["base","v2","ratio"]+VIEWS[m]:   # test 있는 뷰로 (v3 의미 유지하려 v3/v2v3 우선순위는 낮춤)
                if f"{m}_{v}" in TST: sel_v3[m]=v; break
    tk_v3=tkey(sel_v3)
# v3가 하나라도 남아있는지 확인 (전부 다운그레이드되면 v3 시험 의미 없음)
has_v3_left=any("v3" in v for v in sel_v3.values())
w3,a3,_=blend_sel(sel_v3)
if all(k in TST for k in tk_v3.values()) and not DRYRUN_USED:
    pt=sum(w3[m]*runr(TST[tk_v3[m]]) for m in w3 if w3[m]>0 and tk_v3[m] in TST)
    sp=find("sample_submission.csv")
    if sp: s3=pd.read_csv(sp); pc=[c for c in s3.columns if c!=ID_COL][0]; s3[pc]=pt
    else: s3=pd.DataFrame({ID_COL:test[ID_COL] if test is not None else np.arange(len(pt)),"probability":pt})
    s3.to_csv("/kaggle/working/submission_v3_FORCED.csv",index=False)
    print(f"\n💾 submission_v3_FORCED.csv — v3 시험용 (게이트 ❌라도 LB 확인)")
    print(f"   조합 {sel_v3} | 블렌드 OOF={a3:.5f} | v3 잔존={has_v3_left}")
    print(f"   비교 기준: 운영 best LB 0.74209 (= sel_no {sel_no})")
    print(f"   → 이 파일만 LB 제출해서 0.74209 초과 여부 확인. 안 오르면 v3 노이즈 확정.")
    if not has_v3_left: print("   ⚠️ 다운그레이드로 v3가 다 빠짐 — 운영과 동일해질 수 있음(제출 의미 확인)")
else:
    print("v3 강제 제출 skip (test 멤버 부족 또는 DRYRUN)")

v3 강제: test 부재 ['nn_ratio'] → 가용 뷰로 다운그레이드(base 우선, 단 base 부재 시 v2)

💾 submission_v3_FORCED.csv — v3 시험용 (게이트 ❌라도 LB 확인)
   조합 {'lgb': 'v2v3', 'xgb': 'v3', 'cat': 'v2v3', 'lin': 'ratio', 'nn': 'base'} | 블렌드 OOF=0.74067 | v3 잔존=True
   비교 기준: 운영 best LB 0.74209 (= sel_no {'lgb': 'v2', 'xgb': 'base', 'cat': 'v2', 'lin': 'ratio', 'nn': 'ratio'})
   → 이 파일만 LB 제출해서 0.74209 초과 여부 확인. 안 오르면 v3 노이즈 확정.
